# News Modeling

Topic modeling involves **extracting features from document terms** and using
mathematical structures and frameworks like matrix factorization and SVD to generate **clusters or groups of terms** that are distinguishable from each other and these clusters of words form topics or concepts

Topic modeling is a method for **unsupervised classification** of documents, similar to clustering on numeric data

These concepts can be used to interpret the main **themes** of a corpus and also make **semantic connections among words that co-occur together** frequently in various documents

Topic modeling can help in the following areas:
- discovering the **hidden themes** in the collection
- **classifying** the documents into the discovered themes
- using the classification to **organize/summarize/search** the documents

Frameworks and algorithms to build topic models:
- Latent semantic indexing
- Latent Dirichlet allocation
- Non-negative matrix factorization

## Latent Dirichlet Allocation (LDA)
The latent Dirichlet allocation (LDA) technique is a **generative probabilistic model** where each **document is assumed to have a combination of topics** similar to a probabilistic latent semantic indexing model

In simple words, the idea behind LDA is that of two folds:
- each **document** can be described by a **distribution of topics**
- each **topic** can be described by a **distribution of words**

### LDA Algorithm

- 1. For each document, **randomly initialize each word to one of the K topics** (k is chosen beforehand)
- 2. For each document D, go through each word w and compute:
    - **P(T |D)** , which is a proportion of words in D assigned to topic T
    - **P(W |T )** , which is a proportion of assignments to topic T over all documents having the word W
- **Reassign word W with topic T** with probability P(T |D)´ P(W |T ) considering all other words and their topic assignments

![LDA](https://raw.githubusercontent.com/subashgandyer/datasets/main/images/LDA.png)

### Steps
- Install the necessary library
- Import the necessary libraries
- Download the dataset
- Load the dataset
- Pre-process the dataset
    - Stop words removal
    - Email removal
    - Non-alphabetic words removal
    - Tokenize
    - Lowercase
    - BiGrams & TriGrams
    - Lemmatization
- Create a dictionary for the document
- Filter low frequency words
- Create an Index to word dictionary
- Train the Topic Model
- Predict on the dataset
- Evaluate the Topic Model
    - Model Perplexity
    - Topic Coherence
- Visualize the topics

### Install the necessary library

In [3]:
#! pip install pyLDAvis gensim spacy

In [9]:
#pip install requests pandas numpy matplotlib seaborn spacy gensim nltk pyLDAvis

In [17]:
#!pip install --upgrade scipy
#!pip install --upgrade gensim

In [1]:
#!pip install --upgrade numpy

### Import the libraries

In [2]:
import requests
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import spacy

from spacy.lang.en.stop_words import STOP_WORDS
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer

from gensim.models import Phrases
from gensim.models.phrases import Phraser
from gensim.corpora import Dictionary
from gensim.models.ldamodel import LdaModel
from gensim.models import CoherenceModel
from gensim.utils import simple_preprocess
from nltk.corpus import stopwords  # Import stopwords from nltk.corpus
import nltk

# Download the NLTK stopwords if not already downloaded
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\wayne\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### Download the dataset
Dataset: https://raw.githubusercontent.com/subashgandyer/datasets/main/newsgroups.json

#### 20-Newsgroups dataset
- 11K newsgroups posts
- 20 news topics

In [4]:
import requests

url = "https://raw.githubusercontent.com/subashgandyer/datasets/main/newsgroups.json"
response = requests.get(url)
if response.status_code == 200:
    with open("newsgroups.json", "w", encoding="utf-8") as f:
        f.write(response.text)
    print("Dataset downloaded successfully.")
else:
    print("Failed to download the dataset.")

Dataset downloaded successfully.


### Load the dataset

In [6]:
# Download the NLTK stopwords if not already downloaded
nltk.download('stopwords')

# Load the dataset
df = pd.read_json("newsgroups.json")
print("Original DataFrame head:")
print(df.head())

Original DataFrame head:
                                             content  target  \
0  From: lerxst@wam.umd.edu (where's my thing)\nS...       7   
1  From: guykuo@carson.u.washington.edu (Guy Kuo)...       4   
2  From: twillis@ec.ecn.purdue.edu (Thomas E Will...       4   
3  From: jgreen@amber (Joe Green)\nSubject: Re: W...       1   
4  From: jcm@head-cfa.harvard.edu (Jonathan McDow...      14   

            target_names  
0              rec.autos  
1  comp.sys.mac.hardware  
2  comp.sys.mac.hardware  
3          comp.graphics  
4              sci.space  


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\wayne\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [10]:
for idx, text in df['content'].items():
    if idx >= 100:
        break
    raw = str(text).lower()
    print(raw)


from: lerxst@wam.umd.edu (where's my thing)
subject: what car is this!?
nntp-posting-host: rac3.wam.umd.edu
organization: university of maryland, college park
lines: 15

 i was wondering if anyone out there could enlighten me on this car i saw
the other day. it was a 2-door sports car, looked to be from the late 60s/
early 70s. it was called a bricklin. the doors were really small. in addition,
the front bumper was separate from the rest of the body. this is 
all i know. if anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.

thanks,
- il
   ---- brought to you by your neighborhood lerxst ----





from: guykuo@carson.u.washington.edu (guy kuo)
subject: si clock poll - final call
summary: final call for si clock reports
keywords: si,acceleration,clock,upgrade
article-i.d.: shelley.1qvfo9innc3s
organization: university of washington
lines: 11
nntp-posting-host: carson.u.wa

### Preprocess the data

### Email Removal

### Newline Removal

### Single Quotes Removal

### Tokenize
- Create **sent_to_words()** 
    - Use **gensim.utils.simple_preprocess**
    - Use **generator** instead of an usual function

### Stop words Removal
- Extend the stop words corpus with the following words
    - from
    - subject
    - re
    - edu
    - use

#### remove_stopwords( )

In [27]:
from nltk.tokenize import RegexpTokenizer
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# 下載需要的資源
nltk.download('wordnet')
nltk.download('stopwords')

# 初始化斷詞工具、詞形還原器
# 這裡假設你有一個已定義的 tokenizer，例如：
import nltk.tokenize as tokenize
tokenizer = tokenize.RegexpTokenizer(r'\w+')

lemmatizer = WordNetLemmatizer()

# 建立停用詞集合，並擴充額外的詞彙
en_stop = set(stopwords.words('english'))
extra_stopwords = {'from', 'subject', 're', 'edu', 'use'}
en_stop.update(extra_stopwords)

texts = []

for idx, text in df['content'].items():
    # 將文本轉小寫
    raw = str(text).lower()
    
    # 斷詞
    tokens = tokenizer.tokenize(raw)

    # 移除停用詞
    stopped_tokens = [token for token in tokens if token not in en_stop]
    
    # 詞形還原
    lemma_tokens = [lemmatizer.lemmatize(token) for token in stopped_tokens]
    
    # 移除僅包含一個字元的詞彙
    new_lemma_tokens = [token for token in lemma_tokens if len(token) > 1]
    
    # 將處理好的 tokens 加入列表中
    texts.append(new_lemma_tokens)

# 顯示第一筆處理結果
print(texts[0])


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\wayne\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\wayne\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


['lerxst', 'wam', 'umd', 'thing', 'car', 'nntp', 'posting', 'host', 'rac3', 'wam', 'umd', 'organization', 'university', 'maryland', 'college', 'park', 'line', '15', 'wondering', 'anyone', 'could', 'enlighten', 'car', 'saw', 'day', 'door', 'sport', 'car', 'looked', 'late', '60', 'early', '70', 'called', 'bricklin', 'door', 'really', 'small', 'addition', 'front', 'bumper', 'separate', 'rest', 'body', 'know', 'anyone', 'tellme', 'model', 'name', 'engine', 'spec', 'year', 'production', 'car', 'made', 'history', 'whatever', 'info', 'funky', 'looking', 'car', 'please', 'mail', 'thanks', 'il', 'brought', 'neighborhood', 'lerxst']


### Bigrams
- Use **gensim.models.Phrases**
- 100 as threshold

#### make_bigrams( )

In [65]:
from gensim.models import Phrases
from gensim.models.phrases import Phraser

def make_bigrams(texts, threshold=100):
    """
    使用 gensim.models.Phrases 將二元詞組加入文本列表。
    
    參數:
        texts: list of list of tokens，每個元素為一篇文檔的 token 列表
        threshold: Phrases 模型的閾值，數值越大表示組合二元詞的門檻越高，預設為 100
        
    回傳:
        texts_bigrams: 應用二元詞組模型後的文本列表
    """
    bigram = Phrases(texts, threshold=threshold)
    bigram_mod = Phraser(bigram)
    texts_bigrams = [bigram_mod[doc] for doc in texts]
    return texts_bigrams


In [61]:
data_words_bigrams = make_bigrams(texts, threshold=100)

print(data_words_bigrams[0])

['lerxst', 'wam_umd', 'thing', 'car', 'nntp_posting', 'host', 'rac3_wam', 'umd', 'organization', 'university', 'maryland_college', 'park', 'line', '15', 'wondering_anyone', 'could', 'enlighten', 'car', 'saw', 'day', 'door', 'sport', 'car', 'looked', 'late', '60', 'early', '70', 'called', 'bricklin', 'door', 'really', 'small', 'addition', 'front_bumper', 'separate', 'rest', 'body', 'know', 'anyone', 'tellme', 'model', 'name', 'engine', 'spec', 'year', 'production', 'car', 'made', 'history', 'whatever', 'info', 'funky', 'looking', 'car', 'please', 'mail', 'thanks', 'il', 'brought', 'neighborhood', 'lerxst']


### Lemmatization
- Use spacy
    - Download spacy en model (if you have not done that before)
    - Load the spacy model

In [34]:
#! python -m spacy download en

In [51]:
nlp = spacy.load("en_core_web_sm", disable=['parser', 'ner'])

#### lemmatizaton( )

In [53]:
def lemmatization(texts, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV']):
    """https://spacy.io/api/annotation"""
    texts_out = []
    for sent in texts:
        doc = nlp(" ".join(sent)) 
        texts_out.append([token.lemma_ for token in doc if token.pos_ in allowed_postags])
    return texts_out

In [55]:
data_lemmatized = lemmatization(data_words_bigrams, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV'])

In [57]:
print(data_lemmatized[:1])

[['wam_umd', 'thing', 'car', 'nntp_poste', 'host', 'umd', 'organization', 'university', 'park', 'line', 'wondering_anyone', 'enlighten', 'car', 'see', 'day', 'door', 'sport', 'car', 'look', 'late', 'early', 'call', 'door', 'really', 'small', 'addition', 'separate', 'rest', 'body', 'know', 'model', 'name', 'engine', 'spec', 'year', 'production', 'car', 'make', 'history', 'info', 'funky', 'look', 'car', 'mail', 'thank', 'bring', 'neighborhood', 'lerxst']]


### Create a Dictionary

In [59]:
dictionary = Dictionary(texts)

### Create Corpus

### Filter low-frequency words

In [80]:
dictionary.filter_extremes(no_below=10, no_above=0.5)
# convert tokenized documents into a document-term matrix
corpus = [dictionary.doc2bow(text) for text in texts]

In [82]:
print(corpus[0])

[(0, 1), (1, 1), (2, 1), (3, 1), (4, 2), (5, 1), (6, 1), (7, 1), (8, 1), (9, 5), (10, 1), (11, 1), (12, 1), (13, 2), (14, 1), (15, 1), (16, 1), (17, 1), (18, 1), (19, 1), (20, 1), (21, 1), (22, 1), (23, 1), (24, 1), (25, 1), (26, 1), (27, 1), (28, 1), (29, 1), (30, 1), (31, 1), (32, 1), (33, 1), (34, 1), (35, 1), (36, 1), (37, 1), (38, 1), (39, 1), (40, 1), (41, 1), (42, 1), (43, 1), (44, 1), (45, 1), (46, 2), (47, 1), (48, 2), (49, 1), (50, 1), (51, 1)]


### Create Index 2 word dictionary

In [72]:
temp = dictionary[0]  # This is only to "load" the dictionary.
id2word = dictionary.id2token

### Build a News Topic Model

#### LdaModel
- **num_topics** : this is the number of topics you need to define beforehand
- **chunksize** : the number of documents to be used in each training chunk
- **alpha** : this is the hyperparameters that affect the sparsity of the topics
- **passess** : total number of training assess

In [78]:
from gensim.models import LdaModel

# 設定模型參數
num_topics = 10       # 主題數量：你預先定義的主題數量
chunksize = 2000      # 每個訓練區塊中使用的文件數
alpha = 'auto'        # 影響每篇文檔中主題分布的稀疏性，也可以設定為固定數值
passes = 10           # 對整個語料庫進行的訓練遍數

# 建立 LDA 模型
lda_model = LdaModel(corpus=corpus,      # 文檔-詞矩陣（bag-of-words 格式）
                     id2word=id2word,    # 從詞 id 到詞彙的映射字典
                     num_topics=num_topics,
                     chunksize=chunksize,
                     alpha=alpha,
                     passes=passes)

### Print the Keyword in the 10 topics

In [93]:
for i, topic in lda_model.print_topics(num_topics=10, num_words=10):
    print("Topic {}: {}".format(i, topic))

Topic 0: 0.014*"space" + 0.011*"nasa" + 0.011*"00" + 0.010*"10" + 0.009*"25" + 0.008*"15" + 0.007*"gov" + 0.007*"20" + 0.006*"entry" + 0.006*"16"
Topic 1: 0.023*"com" + 0.020*"posting" + 0.019*"nntp" + 0.018*"host" + 0.018*"article" + 0.018*"university" + 0.011*"andrew" + 0.009*"cmu" + 0.009*"reply" + 0.008*"colorado"
Topic 2: 0.012*"god" + 0.010*"one" + 0.009*"people" + 0.008*"would" + 0.006*"think" + 0.006*"say" + 0.005*"christian" + 0.005*"com" + 0.004*"know" + 0.004*"believe"
Topic 3: 0.702*"ax" + 0.053*"max" + 0.017*"g9v" + 0.010*"b8f" + 0.009*"a86" + 0.007*"pl" + 0.007*"75u" + 0.006*"1d9" + 0.005*"3t" + 0.005*"145"
Topic 4: 0.006*"israel" + 0.006*"state" + 0.005*"israeli" + 0.005*"people" + 0.004*"armenian" + 0.004*"year" + 0.004*"article" + 0.004*"would" + 0.004*"mr" + 0.004*"jew"
Topic 5: 0.014*"com" + 0.008*"key" + 0.008*"would" + 0.006*"one" + 0.006*"article" + 0.005*"car" + 0.005*"system" + 0.005*"chip" + 0.005*"get" + 0.004*"state"
Topic 6: 0.010*"game" + 0.009*"team" + 0.0

## Evaluation of Topic Models
- Model Perplexity
- Topic Coherence

### Model Perplexity

Model perplexity is a measurement of **how well** a **probability distribution** or probability model **predicts a sample**

In [109]:
from gensim.models import LdaModel
from gensim.models.coherencemodel import CoherenceModel

# 假設你已經建立好語料庫 corpus 和詞彙映射 id2word，
# 並且已經訓練好 LDA 模型 lda_model

# 計算模型困惑度（log perplexity）
print("Model Log Perplexity: ", lda_model.log_perplexity(corpus))

# 計算主題一致性（例如使用 'c_v' 指標）
coherence_model_lda = CoherenceModel(model=lda_model, texts=texts, dictionary=dictionary, coherence='c_v')

coherence_lda = coherence_model_lda.get_coherence()
print("Topic Coherence: ", coherence_lda)


Model Log Perplexity:  -7.796263572961203
Topic Coherence:  0.6180504553332138


### Topic Coherence
Topic Coherence measures score a single topic by measuring the **degree of semantic similarity** between **high scoring words** in the topic.

In [113]:
from gensim.models.coherencemodel import CoherenceModel

# 假設已經訓練好的 LDA 模型 lda_model，
# texts 是經過預處理後的分詞文本列表，
# dictionary 是 gensim 的 Dictionary 物件

# 使用 'c_v' 作為一致性指標
coherence_model = CoherenceModel(model=lda_model, texts=texts, dictionary=dictionary, coherence='c_v')

# 獲取整個模型的主題一致性分數
overall_coherence = coherence_model.get_coherence()
print("Overall Topic Coherence: ", overall_coherence)

# 如果你想獲取每個主題的分數，可以使用下面方法：
per_topic_coherence = coherence_model.get_coherence_per_topic()
for idx, score in enumerate(per_topic_coherence):
    print("Topic {} Coherence: {:.4f}".format(idx, score))


Overall Topic Coherence:  0.6180504553332138
Topic 0 Coherence: 0.6787
Topic 1 Coherence: 0.6076
Topic 2 Coherence: 0.6274
Topic 3 Coherence: 0.9565
Topic 4 Coherence: 0.5428
Topic 5 Coherence: 0.5304
Topic 6 Coherence: 0.5346
Topic 7 Coherence: 0.6024
Topic 8 Coherence: 0.5364
Topic 9 Coherence: 0.5638


### Visualize the Topic Model
- Use **pyLDAvis**
    - designed to help users **interpret the topics** in a topic model that has been fit to a corpus of text data
    - extracts information from a fitted LDA topic model to inform an interactive web-based visualization

In [115]:
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

# 如果在 Jupyter Notebook 中運行，啟用內嵌視覺化
pyLDAvis.enable_notebook()

# 使用 gensimvis.prepare 生成視覺化資料，參數分別是：
# lda_model：已訓練的 LDA 模型
# corpus：文檔的 bag-of-words 表示
# dictionary：gensim 的 Dictionary 物件
vis_data = gensimvis.prepare(lda_model, corpus, dictionary)

# 在 Jupyter Notebook 中使用 pyLDAvis.display 來顯示視覺化介面
pyLDAvis.display(vis_data)

# 如果不在 Notebook 中，可以使用以下方式保存成 HTML 文件：
pyLDAvis.save_html(vis_data, 'lda_visualization.html')
